# 02 · Weekly database load and rankings

Runs once a week.

1. **Reshape** the task log: team-level rows are collapsed to `Date × Meal`, then the "Branches Missed Tasks" text is split into one row per branch.
2. **Load** new tasks and issues into PostgreSQL. Only rows newer than the latest date already in each table are inserted, so running it twice does no harm.
3. **Rank** every branch and area manager on the last 7 days, using done rate and issue count.

Power BI's weekly, quarterly and yearly reports refresh from the tables this notebook loads.

In production the same steps also run for the Lubda brand (Arabic column names, `lubda.operation` schema). That part is left out here.

In [ ]:
import os
from datetime import datetime, timedelta

import pandas as pd
import psycopg2
import warnings
warnings.filterwarnings("ignore")

## Configuration

Database settings come from environment variables (see `.env.example`), never from the notebook. Set `LOAD_TO_DB=0` to skip the database step and only produce the rankings.

In [ ]:
WORKBOOK_PATH = os.getenv("WORKBOOK_PATH", "../sample_data/sample_workbook.xlsx")
OUTPUT_DIR    = os.getenv("OUTPUT_DIR", "../output")
LOAD_TO_DB    = os.getenv("LOAD_TO_DB", "1") == "1"
REPORT_DATE   = os.getenv("REPORT_DATE", "2026-10-26")   # last day of the week; None -> yesterday

MAIN_DB_CONFIG = {
    "dbname":   os.getenv("PGDATABASE", "shawarma_classic"),
    "user":     os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
    "host":     os.getenv("PGHOST", "localhost"),
    "port":     os.getenv("PGPORT", "5432"),
}

if REPORT_DATE is None:
    REPORT_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 1 · Read and reshape

In [ ]:
tasks = pd.read_excel(WORKBOOK_PATH, sheet_name='Source Data')
issues = pd.read_excel(WORKBOOK_PATH, sheet_name='Reviewing Reports')

tasks.drop(['Team'], axis=1, inplace=True)
tasks['Date'] = pd.to_datetime(tasks['Date']).dt.date
issues['Date'] = pd.to_datetime(issues['Date']).dt.date
issues.drop(['Images'], axis=1, inplace=True)

result = tasks.groupby(['Date', 'Meal']).agg({
    'Done Tasks': 'sum',
    'Missed Tasks': 'sum',
    'Branches Missed Tasks': lambda x: '\n'.join(x.dropna().astype(str)),
    'Due time': 'first'
}).reset_index()

# 1. Split the column into lists (if it's not already)
result['Branches Missed Tasks'] = result['Branches Missed Tasks'].str.split('\n')

# 2. Explode the list into separate rows
result = result.explode('Branches Missed Tasks')

# 3. (Optional) Strip spaces just in case
result['Branches Missed Tasks'] = result['Branches Missed Tasks'].str.strip()

result.head(10)

## Step 2 · Incremental load into PostgreSQL

In [ ]:
def connect_to_postgres(config):
    """Connect to PostgreSQL and return the connection and cursor."""
    conn = psycopg2.connect(**config)
    return conn, conn.cursor()

def insert_new_records(main_cursor, table_name, data, query):
    """Insert records newer than the latest date already in the table."""
    # Fetch the latest date in the main database
    main_cursor.execute(f'SELECT MAX("date") FROM training.{table_name}')
    latest_date = main_cursor.fetchone()[0]
    print(f"The latest date in {table_name}: {latest_date}")

    # Filter new data
    if latest_date:
        new_data = data[data['Date'] > latest_date]
    else:
        new_data = data  # Insert all data if the database is empty

    for row in new_data.itertuples(index=False):
        values = tuple(v.item() if hasattr(v, 'item') else v for v in row)   # numpy -> Python types
        main_cursor.execute(query, values)

    print(f"{len(new_data)} new records added to the {table_name} table.")


issues_data = issues[['Date', 'Branch', 'Meal', 'Description', 'Manager']]
tasks_data = result.copy()

if not LOAD_TO_DB:
    print("LOAD_TO_DB=0, skipping the database step.")
else:
    main_conn, main_cursor = connect_to_postgres(MAIN_DB_CONFIG)
    try:
        issues_query = """
            INSERT INTO training.issues ("date", "branch", "meal", "description", "manager")
            VALUES (%s, %s, %s, %s, %s)
        """
        insert_new_records(main_cursor, "issues", issues_data, issues_query)

        tasks_query = """
            INSERT INTO training.tasks ("date", "meal", "done_tasks", "missed_tasks", "branch_missed", "due_time")
            VALUES (%s, %s, %s, %s, %s, %s)
        """
        insert_new_records(main_cursor, "tasks", tasks_data, tasks_query)

        # Both tables are committed together, or not at all
        main_conn.commit()
        print("All data inserted successfully.")
    except Exception as e:
        main_conn.rollback()
        print(f"Error during data insertion, rolled back: {e}")
        raise
    finally:
        main_cursor.close()
        main_conn.close()

## Step 3 · Branch and manager weekly rankings

Window: the 7 days ending on `REPORT_DATE`.

- **Done rate** = `100 − missed / tasks_in_window × 100`. Branches with no misses get 100.
- **Branch score** = `0.90 × done_rate_norm + 0.10 × (1 − issues_norm)`
- **Manager score** = `0.85 × done_rate_norm + 0.15 × (1 − issues_per_branch_norm)`
- Rank 1 = best. Branch grades run from *Excellent* (> 0.95) to *Terrible* (≤ 0.70).

In [ ]:
branches = pd.read_excel(WORKBOOK_PATH, sheet_name='branches and codes')
managers = pd.read_excel(WORKBOOK_PATH, sheet_name='Managers')

start = (pd.Timestamp(REPORT_DATE) + timedelta(days=1)).date()   # exclusive
end = start - timedelta(days=7)
result_11 = result[(result['Date'] >= end) & (result['Date'] < start)]\
                            .sort_values(by=['Date', 'Due time', 'Meal'], ascending=[False, True, True])

issues_11 = issues[(issues['Date'] >= end) & (issues['Date'] < start)]\
                            .sort_values(by='Date', ascending=False)
print(f"Window: {end} to {start - timedelta(days=1)}")


tasks_n = result_11[['Date', 'Meal', 'Due time']].drop_duplicates().sort_values(by=['Meal', 'Due time']).shape[0]
branches_done_rates = 100 - round((result_11[result_11['Branches Missed Tasks'] != ''].groupby(['Branches Missed Tasks'])['Meal'].count().sort_values() / tasks_n) * 100, 1)


all_branches_done_rates = branches.reset_index().merge(branches_done_rates, left_on='Branch', right_on='Branches Missed Tasks', how='left').set_index('Branch').fillna(100)
all_branches_done_rates.rename(columns={'Meal': 'Done Rate'}, inplace=True)
all_branches_done_rates.drop('index', axis=1, inplace=True)


all_branches_done_rates = all_branches_done_rates.reset_index().merge(managers, left_on='Branch', right_on='Branch', how='left').set_index('Branch')


issues_11 = issues_11[['Code', 'Meal', 'Description']]
tasks_issues = all_branches_done_rates.reset_index().merge(issues_11, left_on='Code', right_on='Code', how='left')
tasks_issues.drop(['Code'], axis=1, inplace=True)


summary = (
    tasks_issues
    .groupby(['Manager', 'Branch'])[['Done Rate', 'Meal']]
    .agg({'Done Rate': 'mean', 'Meal': 'count'})
    .rename(columns={'Meal': 'Number of Issues'})
    .reset_index()
).drop_duplicates(['Branch', 'Done Rate'])

summary['Done Rate'] = summary['Done Rate'].round(1)

# Normalize the metrics (so Done Rate and Number of Issues are comparable)
summary['Done Rate_norm'] = summary['Done Rate'] / summary['Done Rate'].max()
summary['Issues_norm'] = summary['Number of Issues'] / summary['Number of Issues'].max()

# Compute a combined score:
#   Higher Done Rate (good) -> positive
#   Lower Issues (good) -> negative
summary['Performance Score'] = (0.9 * summary['Done Rate_norm']) + (0.1 * (1 - summary['Issues_norm']))

# Rank: 1 = best
summary['Branches Rank'] = summary['Performance Score'].rank(ascending=False, method='min').astype(int)


# Create grades based on the combined performance
grade_bins = [0, 0.7, 0.8, 0.9, 0.95, 1.0]
grade_labels = ['Terrible', 'Bad', 'Fair', 'Good', 'Excellent']

summary['Completion Grade'] = pd.cut(
    summary['Performance Score'],
    bins=grade_bins,
    labels=grade_labels
)


# Clean up
summary = summary.drop(columns=['Done Rate_norm', 'Issues_norm', 'Performance Score']).sort_values('Branches Rank')

summary.to_excel(os.path.join(OUTPUT_DIR, 'Branches Weekly Performance Summary.xlsx'), index=False)


# Calculate average performance per branch for each manager
manager_summary = (
    tasks_issues
    .groupby(['Manager', 'Branch'])[['Done Rate', 'Meal']]
    .agg({'Done Rate': 'mean', 'Meal': 'count'})
    .rename(columns={'Meal': 'Issues per Branch'})
    .reset_index()
    .groupby('Manager')
    .agg({
        'Done Rate': 'mean',
        'Issues per Branch': 'mean',
        'Branch': 'count'
    })
    .rename(columns={'Branch': 'Number of Branches'})
    .reset_index()
)

manager_summary['Number of Issues'] = manager_summary['Issues per Branch'] * manager_summary['Number of Branches']
manager_summary['Done Rate'] = manager_summary['Done Rate'].round(1)
manager_summary['Issues per Branch'] = manager_summary['Issues per Branch'].round(1)

# Normalize the metrics (so Done Rate and Issues per Branch are comparable)
manager_summary['Done Rate_norm'] = manager_summary['Done Rate'] / manager_summary['Done Rate'].max()
manager_summary['Issues_norm'] = manager_summary['Issues per Branch'] / manager_summary['Issues per Branch'].max()

# Compute a combined score:
#   Higher Done Rate (good) -> positive
#   Lower Issues (good) -> negative
manager_summary['Performance Score'] = (0.85 * manager_summary['Done Rate_norm']) + (0.15 * (1 - manager_summary['Issues_norm']))

# Rank: 1 = best
manager_summary['Managers Rank'] = manager_summary['Performance Score'].rank(ascending=False, method='min').astype(int)

# Clean up
manager_summary = manager_summary.drop(columns=['Done Rate_norm', 'Issues_norm', 'Performance Score', 'Number of Branches']).sort_values('Managers Rank')
manager_summary.to_excel(os.path.join(OUTPUT_DIR, 'Managers Weekly Performance Summary.xlsx'), index=False)

### Branches

In [ ]:
summary

### Managers

In [ ]:
manager_summary